# M115 Lab 1: Phishing URL Detection — EDA, Parameterized Logistic Regression & SHAP Explainability

---

## 🏛️ ENSIASD Taroudant — Ibn Zohr University
**Master in Intelligent Systems and Cyber Defense (Module M115: AI & Intelligent Systems)**  
**Instructor:** Prof. Sara EL-ATEIF | **Academic Year:** 2026–2027

---

## 🧭 Lab Overview & Solution Key

This notebook contains the assignment of **Lab 1** on the **[PhiUSIIL Phishing URL Dataset](https://www.kaggle.com/datasets/kaggleprollc/phishing-url-websites-dataset-phiusiil/data)** (*235,795 URLs and 56 lexical, network, and HTML features*, Prasad & Chandra, *Computers & Security*, 2024).

### 📋 Table of Contents
* **1 - Importing Packages & Loading Data**
* **2 - Part 1: Security-Oriented Exploratory Data Analysis (EDA)**
  * 2.1 Data Quality Inspection & SOC Target Alignment (Exercise 1)
  * 2.2 Class Balance, Numerical Distributions & Binary Security Indicators (Exercise 2)
  * 2.3 Identifier Cleanup & Feature–Target Correlations (Exercise 3)
  * *Part 1 Reflection Questions*
* **3 - Part 2: Zero-Data Leakage Preprocessing & Standardization**
  * 3.1 Stratified 75/25 Split & Training-Only StandardScaler (Exercise 4)
  * *Part 2 Reflection Questions*
* **4 - Part 3: Parameterized Logistic Regression & Interpretability**
  * 4.1 Comparing 5 Logistic Regression Configurations (Exercise 5)
  * 4.2 L1 Regularization Path & Feature Selection (Exercise 6)
  * 4.3 Interpreting Coefficients ($w_j$) and Odds Ratios ($e^{w_j}$) (Exercise 7)
  * *Part 3 Reflection Questions*
* **5 - Part 4: Explainable AI (SHAP) & SOC Threshold Analysis**
  * 5.1 Global Explanations with `shap.LinearExplainer` (Exercise 8)
  * 5.2 Local SOC Alert Triage: TP, TN, and FP Waterfall Plots (Exercise 9)
  * 5.3 Operational Decision Threshold Tuning ($\tau \in \{0.20, 0.50, 0.85\}$) (Exercise 10)
  * *Part 4 Reflection Questions*
* **6 - Conclusion & SOC Engineering Synthesis**

---
## 1 - Importing Packages & Loading Data

Run the cell below to import the required libraries and configure reproducibility (`SEED = 42`):

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, average_precision_score
)
import shap

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4.5)
SEED = 42
np.random.seed(SEED)

df = pd.read_csv("Desktop/PhiUSIIL_Phishing_URL_Dataset.csv")
pd.set_option("display.max_columns", None)
print(df.shape)
df.head()


---
## 2 - Part 1: Security-Oriented Exploratory Data Analysis (EDA)

### 2.1 Data Quality Inspection & SOC Target Alignment (Exercise 1)

First, we load `PhiUSIIL_Phishing_URL_Dataset.csv` and inspect its shape, data types, missing values, duplicate rows, and summary statistics across key numerical features (`URLLength`, `LineOfCode`, `CharContinuationRate`, `NoOfSelfRef`).

In [ ]:
# Load dataset and inspect data quality
df = pd.read_csv("Desktop/PhiUSIIL_Phishing_URL_Dataset.csv")
print("Dataset Shape:", df.shape)
print("Total Missing Values:", df.isna().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())
print("\nDataset Info:")
df.info()

display(df[['FILENAME', 'URL', 'URLLength', 'LineOfCode', 'IsHTTPS', 'HasPasswordField', 'NoOfSelfRef', 'label']].head(5))
display(df[['URLLength', 'LineOfCode', 'CharContinuationRate', 'NoOfSelfRef']].describe().round(2))

> **⚠️ Critical Cyber Label Warning in PhiUSIIL:**  
> In the raw CSV file, **`label = 1` denotes Legitimate (Benign)** and **`label = 0` denotes Phishing (Malicious)**.  
> In a SOC, the positive class (`1`) **must always represent the attack** (`Phishing = 1`, `Benign = 0`) so that True Positives (TP), Precision, and Recall measure our threat detection capability.

### Exercise 1

In [ ]:
# EXERCISE 1: Data Quality & SOC Target Alignment (1 = Phishing, 0 = Benign)

### START CODE HERE ### (≈ 2 lines of code)
df['IsPhishing'] = 1 - df['label']
phishing_ratio = df['IsPhishing'].mean()
### END CODE HERE ###

print(f"Phishing URLs (1): {(df['IsPhishing'] == 1).sum():,}")
print(f"Legitimate URLs (0): {(df['IsPhishing'] == 0).sum():,}")
print(f"Phishing Proportion: {phishing_ratio * 100:.2f}%")
print("\nNormalized Class Proportions:")
print(df['IsPhishing'].value_counts(normalize=True).round(4))

In [ ]:
INTERPRETATION
- Phishing: 100,945 URLs (42.81%). Legitimate: 134,850 URLs (57.19%).
- Check: 100,945 + 134,850 = 235,795, exactly the number of rows.
- The classes are moderately balanced, but accuracy alone can still mislead, so Precision and Recall are used.

---
### 2.2 Class Balance, Numerical Distributions & Binary Security Indicators (Exercise 2)

We compare legitimate (`0`) and phishing (`1`) URLs across class balance, continuous lexical/HTML features (`URLLength`, `CharContinuationRate`, `NoOfSelfRef`), and binary security indicators (`IsHTTPS`, `HasPasswordField`, `HasCopyrightInfo`, `HasSocialNet`).

### Exercise 2

In [ ]:
# EXERCISE 2: Visualizing Distributions and Binary Security Indicators
binary_cols = ['IsHTTPS', 'HasPasswordField', 'HasCopyrightInfo', 'HasSocialNet']
num_cols = ['URLLength', 'CharContinuationRate', 'NoOfSelfRef']

### START CODE HERE ### (≈ 1 line of code)
binary_stats = df.groupby('IsPhishing')[binary_cols].mean() * 100
### END CODE HERE ###

print("Percentage (%) of URLs presenting each binary indicator per class (0=Benign, 1=Phishing):")
display(binary_stats.round(2))

# 1. Class Balance & Binary Indicators Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.countplot(data=df, x='IsPhishing', hue='IsPhishing', ax=axes[0], palette=['#16a34a', '#dc2626'], legend=False)
axes[0].set_title("Class Balance (0 = Legitimate, 1 = Phishing)")
axes[0].set_xlabel("IsPhishing (0 = Benign, 1 = Phishing)")
axes[0].set_ylabel("URL Count")

binary_stats.T.plot(kind='bar', ax=axes[1], color=['#16a34a', '#dc2626'])
axes[1].set_title("Prevalence (%) of Binary Security Indicators by Class")
axes[1].set_ylabel("Percentage (%)")
axes[1].set_xticklabels(binary_cols, rotation=15)
axes[1].legend(["Legitimate (0)", "Phishing (1)"])
plt.tight_layout()
plt.show()

# 2. Numerical Distributions (KDE Plots clipped at 99th percentile for readability)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(num_cols):
    upper_clip = df[col].quantile(0.99)
    sns.kdeplot(
        data=df[df[col] <= upper_clip],
        x=col,
        hue='IsPhishing',
        common_norm=False,
        fill=True,
        palette=['#16a34a', '#dc2626'],
        ax=axes[i]
    )
    axes[i].set_title(f"Distribution of {col} (<=99th pct)")
plt.tight_layout()
plt.show()

In [ ]:
INTERPRETATION
- IsHTTPS at 100% for legitimate sites partly reflects this dataset. HTTPS alone does not prove a site is safe, since about half of the phishing sites also use it.
- The curves are clipped at the 99th percentile to keep them readable.

---
### 2.3 Identifier Cleanup & Feature–Target Correlations (Exercise 3)

We drop non-generalizable raw text strings (`FILENAME`, `URL`, `Domain`, `TLD`, `Title`), the old `label`, and the pre-computed `URLSimilarityIndex` score so our model learns from genuine lexical and HTML indicators. Then we compute the Pearson correlation of each numerical feature with `IsPhishing`.

### Exercise 3

In [ ]:
# EXERCISE 3: Constructing X, y and Analyzing Target Correlations
cols_to_drop = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title', 'label', 'IsPhishing', 'URLSimilarityIndex']

### START CODE HERE ### (≈ 3 lines of code)
X = df.drop(columns=cols_to_drop)
y = df['IsPhishing']
correlations = X.corrwith(y).sort_values()
### END CODE HERE ###

print("Shape of X:", X.shape, "| Shape of y:", y.shape)

# Plot Top 15 Negatively and Top 15 Positively Correlated Features
top_corr = pd.concat([correlations.head(15), correlations.tail(15)])
plt.figure(figsize=(10, 7.5))
colors = ['#16a34a' if c < 0 else '#dc2626' for c in top_corr.values]
top_corr.plot(kind='barh', color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title("Top 15 Negatively (Benign) and Positively (Phishing) Correlated Features with IsPhishing")
plt.xlabel("Pearson Correlation Coefficient (r)")
plt.tight_layout()
plt.show()

In [ ]:
INTERPRETATION
- The negative side is stronger than the positive side (about -0.78 vs +0.54), so the HTML features separate the classes better than the URL text features.
- Correlation is not causation, and Pearson only measures linear relationships.
- Many features overlap, so correlations should be read with the model weights and SHA

---
### 💡 Part 1 Reflection Questions

1. **Looking at your plot for `IsHTTPS`, what percentage of phishing sites currently use an HTTPS certificate? Why is a static rule *"If HTTPS then Legitimate"* dangerous?**
   * **Answer:** 49.22%, from the table in Exercise 2.
   By adopting the rule “If HTTPS then Legitimate” dangerous we would miss half the attacks. About 49% of the phishing sites in our data use HTTPS. The rule would label all of these sites as legitimate and thus a huge share of the threats would slip through. So it is not to be used as a single indicator .

2. **Why do phishing websites typically have very few internal links (`NoOfSelfRef`) but very frequently contain a password field (`HasPasswordField`)?**
   * **Answer:** phishing pages are usually a single copied page built for one purpose which is to gain acces or steal credentials, so there is almost nothing else to link to. 
   As the goal of a phishing page is to steal credentials. It imitates a login page and includes a password field so the victim types their password, which is then sent to the attacker.

3. **What would happen in production if we left the `FILENAME`, `URL`, or `Domain` column in our training matrix?**
   * **Answer:** these columns are raw text and almost every value is unique . The model would memorize specific strings instead of learning general patterns (overfitting), it would score very well on training data but fail on new URLs that never been seen.

---
## 3 - Part 2: Zero-Data Leakage Preprocessing & Standardization

### 3.1 Stratified 75/25 Split & Training-Only StandardScaler (Exercise 4)

We apply the **SecMLOps Golden Rule**: split the dataset into training (75%) and test (25%) subsets *before* computing Z-score normalization parameters ($\mu_{\text{train}}, \sigma_{\text{train}}$).

### Exercise 4

In [ ]:
# EXERCISE 4: Stratified Split and Leakage-Free Standardization

### START CODE HERE ### (≈ 4 lines of code)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
### END CODE HERE ###

# Retain column names and indices for weight and SHAP interpretability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

print("Train shape:", X_train_scaled.shape, "| Test shape:", X_test_scaled.shape)
print("Mean of X_train_scaled (URLLength):", round(X_train_scaled['URLLength'].mean(), 4))
print("Std of X_train_scaled (URLLength):", round(X_train_scaled['URLLength'].std(), 4))

In [ ]:
INTERPRETATION
- Scaling is needed because L1 and L2 penalties depend on feature scale.
- Fitting the scaler on all the data would leak test information and make the results too optimistic.

---
### 💡 Part 2 Reflection Questions

1. **Compare the scale of `LineOfCode` (values up to 10,000+) and `IsHTTPS` (0 or 1) before standardization. Why would L1/L2 regularization be skewed without `StandardScaler`?**
   * **Answer:**LineOfCode has huge values (442,666), while IsHTTPS is only 0 or 1. Regularization penalizes the size of the weights. LineOfCode needs only a tiny weight to have an effect, so it is barely penalized. IsHTTPS needs a much larger weight, so it is penalized heavily. The penalty therefore depends on the feature’s scale and not on how useful it is: L1 could even shrink IsHTTPS to zero. After standardization, all features are on the same scale, so the penalty is fair and the weights are comparable.

2. **Precisely explain why calling `scaler.fit_transform(X)` before `train_test_split` constitutes Preprocessing Data Leakage.**
   * **Answer:** if we follow that order fit_transform(X) will compute the mean and standard deviation of each column using all rows, including the rows that will later become the test set. Then the split happens using train_test_split , in this case the test set influences the training data as The mean and std used to scale the training rows contain information from the test rows and thus the test set is no longer truly unseen which is a data leakage.  

---
## 4 - Part 3: Parameterized Logistic Regression & Interpretability

### 4.1 Comparing 5 Logistic Regression Configurations (Exercise 5)

We train and evaluate the five specified Logistic Regression configurations on `X_train_scaled` and `X_test_scaled`:
* **M1: Unregularized (`penalty=None`)**
* **M2: Strong L2 Ridge (`penalty='l2', C=0.001`)**
* **M3: Standard L2 Ridge (`penalty='l2', C=1.0`)**
* **M4: Sparse L1 Lasso (`penalty='l1', C=0.01`)**
* **M5: SOC Weighted (`penalty='l2', C=1.0, class_weight='balanced'`)**

### Exercise 5

In [ ]:
# EXERCISE 5: Training and Comparing 5 Logistic Regression Configurations
configurations = {
    "M1: Unregularized (None)": LogisticRegression(penalty=None, solver='lbfgs', max_iter=500, random_state=SEED),
    "M2: Strong L2 Ridge (C=0.001)": LogisticRegression(penalty='l2', C=0.001, solver='lbfgs', max_iter=500, random_state=SEED),
    "M3: Standard L2 Ridge (C=1.0)": LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=500, random_state=SEED),
    "M4: Sparse L1 Lasso (C=0.01)": LogisticRegression(penalty='l1', C=0.01, solver='liblinear', max_iter=500, random_state=SEED),
    "M5: SOC Weighted (balanced)": LogisticRegression(penalty='l2', C=1.0, class_weight='balanced', solver='lbfgs', max_iter=500, random_state=SEED),
}

results = []
for name, model in configurations.items():
    model.fit(X_train_scaled, y_train)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_prob >= 0.50).astype(int)
    w = model.coef_[0]
    n_zero_weights = int(np.sum(np.abs(w) < 1e-4))

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    results.append({
        "Model": name,
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall (TPR)": round(recall_score(y_test, y_pred), 4),
        "F1-Score": round(f1_score(y_test, y_pred), 4),
        "PR-AUC": round(average_precision_score(y_test, y_prob), 4),
        "False Alarms (FP)": int(fp),
        "Missed Phish (FN)": int(fn),
        "Zeroed Weights (Sparsity)": f"{n_zero_weights} / {len(w)}"
    })

results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
INTERPRETATION
- Strong L2 hurts performance, while a mild one doesn’t matter.
- L1 gives a lighter model with half the features at almost no loss in quality.
- Class weighting would matter more if phishing were rare.

---
### 4.2 L1 Regularization Path & Feature Selection (Exercise 6)

We vary `C` across `[0.002, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]` with `penalty='l1'` (`solver='liblinear'`) and plot the number of retained features ($|w_j| \ge 10^{-4}$) as a function of `C` on a logarithmic scale.

### Exercise 6

In [ ]:
# EXERCISE 6: L1 Regularization Path (Number of Active Features vs. C)
import warnings
warnings.filterwarnings("ignore")   # hides the red deprecation warnings

C_values = [0.002, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]
active_features_count = []

# Random sample of 10,000 training rows (same rows for every C, to keep it fast)
X_tr_s = X_train_scaled.sample(n=10000, random_state=SEED)
y_tr_s = y_train.loc[X_tr_s.index]

for c_val in C_values:
    ### START CODE HERE ### (≈ 3 lines of code)
    m_l1 = LogisticRegression(penalty='l1', C=c_val, solver='liblinear', max_iter=50, random_state=SEED)
    m_l1_fit = m_l1.fit(X_tr_s, y_tr_s)
    n_active = int(np.sum(np.abs(m_l1_fit.coef_[0]) >= 1e-4))
    ### END CODE HERE ###
    print("C =", c_val, "-> active features:", n_active)
    active_features_count.append(n_active)

path_df = pd.DataFrame({"C": C_values, "Active_Features": active_features_count, "Zeroed_Features": [X.shape[1] - k for k in active_features_count]})
display(path_df)

plt.figure(figsize=(8, 4))
plt.plot(C_values, active_features_count, marker='o', linewidth=2, color='#0284c7')
plt.xscale('log')
plt.xlabel("Inverse Regularization Strength C (log scale)")
plt.ylabel("Number of Retained Features (|w_j| >= 1e-4)")
plt.title(f"L1 Lasso Regularization Path (Total Features = {X.shape[1]})")
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
INTERPRETATION
- The path was computed on a 10,000-row sample instead of all 176,846 rows for speed.
- Exact counts differ slightly from Exercise 5, but the trend is the same.

---
### 4.3 Interpreting Coefficients ($w_j$) and Odds Ratios ($e^{w_j}$) (Exercise 7)

We extract the learned log-odds weights $w_j$ from `M3: Standard L2 Ridge (C=1.0)`, compute the Odds Ratio $e^{w_j}$, compare against `M4: Sparse L1 Lasso (C=0.01)`, and display the **Top 8 Phishing** ($w_j > 0$) and **Top 8 Benign** ($w_j < 0$) features.

### Exercise 7

In [ ]:
# EXERCISE 7: Inspecting Log-Odds (w_j) and Odds Ratios (exp(w_j))
model_l2 = configurations["M3: Standard L2 Ridge (C=1.0)"]
model_l1 = configurations["M4: Sparse L1 Lasso (C=0.01)"]

### START CODE HERE ### (≈ 4 lines of code)
weights_df = pd.DataFrame({
    "Feature": X.columns,
    "Weight (w_j)": model_l2.coef_[0],
    "Odds Ratio exp(w_j)": np.exp(model_l2.coef_[0]),
    "L1_Sparse_Weight": model_l1.coef_[0]
}).sort_values("Weight (w_j)", ascending=False)
### END CODE HERE ###

print("🔴 TOP 8 FEATURES PUSHING TOWARD PHISHING (w_j > 0):")
display(weights_df.head(8).round(4))

print("\n🟢 TOP 8 FEATURES PUSHING TOWARD BENIGN (w_j < 0):")
display(weights_df.tail(8).round(4))

In [ ]:
INTERPRETATION
- Strong phishing signals: URL character ratios.
- Strong legitimate signals: HTTPS, JavaScript, internal links, social links.
- Features dropped by L1 are redundant with others.

---
### 💡 Part 3 Reflection Questions —

1. **How many variables did `M4: Sparse L1 Lasso (C=0.01)` eliminate (weights set to zero)? Did the F1-Score drop significantly? What is the advantage for a real-time firewall?**
   * **Answer:** M4 eliminated 24 of the 49 variables . The F1-Score did not drop significantly: it went from 0.9992 (M3) to 0.9982 (M4), a loss of only about 0.001. For a real-time firewall, a model with half the features is faster and lighter, because fewer values must be extracted and computed for each URL. It is also easier to explain and less prone to overfitting.


2. **What is the impact of `class_weight='balanced'` (M5) on False Positives and False Negatives compared to M3?**
   * **Answer:** M5 gave 13 False Positives and 25 False Negatives, against 10 and 28 for M3. Balanced weights penalize missed phishing more, so M5 missed 3 fewer phishing URLs but raised 3 more false alarms. The effect is very small, because the classes are only moderately imbalanced (43% phishing vs 57% legitimate). It would matter much more if phishing were rare.

3. **How do you interpret a positive weight $w_j > 0$ versus a negative weight $w_j < 0$ in the formula $\sigma(\mathbf{w}^\top\mathbf{x} + b)$?**
   * **Answer:** The term $\mathbf{w}^\top\mathbf{x} + b$ is the log-odds of phishing, and the sigmoid $\sigma$ turns it into a probability. A positive weight $w_j > 0$ increases the log-odds when the feature increases, so it pushes the prediction toward phishing (odds ratio $e^{w_j} > 1$). A negative weight $w_j < 0$ decreases the log-odds, so it pushes toward legitimate (odds ratio < 1). The larger the absolute value, the stronger the influence. Since the features are standardized, weights are comparable across features.

---
## 5 - Part 4: Explainable AI (SHAP) & SOC Threshold Analysis

### 5.1 Global Explanations with `shap.LinearExplainer` (Exercise 8)

We initialize `shap.LinearExplainer` on `model_l2` (`M3`) using `X_train_scaled` as the background reference distribution, compute exact Shapley values on the first `1,000` test URLs, and plot the global Bar and Beeswarm charts (`max_display=15`).

### Exercise 8

In [ ]:
# EXERCISE 8: Computing SHAP Values and Global Importance Plots (1,000 Test URLs)
X_test_sample = X_test_scaled.iloc[:1000]

### START CODE HERE ### (≈ 2 lines of code)
explainer = shap.LinearExplainer(model_l2, X_train_scaled)
shap_values = explainer(X_test_sample)
### END CODE HERE ###

# 1. Global Mean Absolute SHAP Importance (Bar Plot)
shap.plots.bar(shap_values, max_display=15)

# 2. Global Directional Impact (Beeswarm Plot)
shap.plots.beeswarm(shap_values, max_display=15)

In [ ]:
INTERPRETATION
- The model relies mainly on IsHTTPS and the URL character ratios.
- HTTPS, JavaScript, internal links and social links push toward legitimate. High letter, digit and special-character ratios push toward phishing.

---
### 5.2 Local SOC Alert Triage: TP, TN, and FP Waterfall Plots (Exercise 9)

We locate the first **True Positive (TP)**, **True Negative (TN)**, and **False Positive (FP — False Alarm)** in our 1,000-URL test sample and inspect their local SHAP Waterfall plots (`max_display=10`), passing unscaled `X_test` values in `data=` so SOC analysts can read actual URL metrics.

### Exercise 9

In [ ]:
# EXERCISE 9: Local SOC Triage of TP, TN, and FP Cases with SHAP Waterfall
y_test_sample = y_test.iloc[:1000].values
y_prob_sample = model_l2.predict_proba(X_test_sample)[:, 1]
y_pred_sample = (y_prob_sample >= 0.50).astype(int)

### START CODE HERE ### (≈ 3 lines of code)
tp_idx = None
tn_idx = None
fp_candidates = None
### END CODE HERE ###

fp_idx = int(fp_candidates[0]) if len(fp_candidates) > 0 else int(np.argmax(y_prob_sample * (y_test_sample == 0)))

def plot_soc_waterfall(idx, case_title):
    print(f"=== {case_title} (Sample Index={idx}) | True Label={y_test_sample[idx]} | P(Phishing)={y_prob_sample[idx]*100:.2f}% ===")
    local_exp = shap.Explanation(
        values=shap_values.values[idx],
        base_values=float(np.ravel(shap_values.base_values)[idx]),
        data=X_test.iloc[idx].values, # Unscaled values for human analyst readability
        feature_names=X.columns
    )
    shap.plots.waterfall(local_exp, max_display=10)

plot_soc_waterfall(tp_idx, "CASE 1: TRUE POSITIVE (Confirmed Phishing Attack Caught)")
plot_soc_waterfall(tn_idx, "CASE 2: TRUE NEGATIVE (Legitimate Traffic Correctly Cleared)")
plot_soc_waterfall(fp_idx, "CASE 3: FALSE POSITIVE (Legitimate Site Wrongly Flagged — False Alarm)")

In [ ]:
INTERPRETATION
- The TP and TN show very confident, consistent decisions.
- The closest false alarm was driven by missing HTML elements (social links, copyright, description, title match), not by anything malicious in the URL.

---
### 5.3 Operational Decision Threshold Tuning ($\tau \in \{0.20, 0.50, 0.85\}$) (Exercise 10)

We evaluate `model_l2` (`M3`) on the full test set across three operational decision thresholds:
* **$\tau = 0.20$ (Threat Hunting Mode):** Maximizes Recall to catch stealthy phishing campaigns.
* **$\tau = 0.50$ (Default Mode):** Balanced threshold.
* **$\tau = 0.85$ (Automated Firewall Blocking Mode):** Maximizes Precision and minimizes False Positives to prevent blocking legitimate business traffic.

### Exercise 10

In [ ]:
# EXERCISE 10: Comparing Operational Decision Thresholds (0.20, 0.50, 0.85)
y_prob_full = model_l2.predict_proba(X_test_scaled)[:, 1]
thresholds = [0.20, 0.50, 0.85]
threshold_records = []

for tau in thresholds:
    ### START CODE HERE ### (≈ 2 lines of code)
    y_pred_tau = (y_prob_full >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_tau).ravel()
    ### END CODE HERE ###

    threshold_records.append({
        "Threshold (tau)": tau,
        "SOC Operational Mode": "Threat Hunting" if tau == 0.20 else ("Default" if tau == 0.50 else "Automated Blocking"),
        "Precision": round(precision_score(y_test, y_pred_tau), 4),
        "Recall (TPR)": round(recall_score(y_test, y_pred_tau), 4),
        "F1-Score": round(f1_score(y_test, y_pred_tau), 4),
        "True Positives (TP)": int(tp),
        "True Negatives (TN)": int(tn),
        "False Positives (FP)": int(fp),
        "False Negatives (FN)": int(fn)
    })

threshold_df = pd.DataFrame(threshold_records)
display(threshold_df)

In [ ]:
INTERPRETATION
- The threshold is an operational choice, not a property of the model.
- A SOC can use two thresholds: automatic blocking at 0.85, and an analyst queue between 0.20 and 0.85.

---
### 💡 Part 4 Reflection Questions —

1. **What is the mathematical difference between inspecting a global weight $w_j$ and looking at a local SHAP value $\phi_j(x)$ for a specific URL?**
   * **Answer:** The weight $w_j$ is a single fixed number learned by the model: it is the same for every URL and says how much the log-odds of phishing change when feature $j$ increases by one unit (one standard deviation here). The SHAP value $\phi_j(x)$ is specific to one URL. For a linear model it equals $w_j \cdot (x_j - E[x_j])$, so it depends on how far this URL's value is from the average. The SHAP values of a URL add up to its prediction: $f(x) = E[f(X)] + \sum_j \phi_j(x)$. So $w_j$ explains the model globally, while $\phi_j(x)$ explains why this particular URL received its score.

2. **On your False Positive (FP) Waterfall plot, which 2 or 3 features misled the model and caused this false alarm?**
   * **Answer:** The features that pushed this legitimate site toward phishing were `HasSocialNet` = 0 (+1.95), `HasCopyrightInfo` = 0 (+1.21) and `DomainTitleMatchScore` = 0 (+1.10). This site lacks elements that most legitimate sites have, so the model read the absence as suspicious. Note that its P(Phishing) was 31.58%, below the 0.50 threshold, so it was not actually flagged: the test sample contained no real false positive, and this case is the legitimate URL with the highest phishing probability.

3. **Why would a SOC deploy $\tau = 0.85$ for automated DNS/firewall blocking while routing URLs with $0.20 \le P(\text{Phishing}) < 0.85$ to a human analyst queue?**
   * **Answer:** Automatic blocking acts without human review, so a false positive blocks a legitimate site and disrupts business traffic. A high threshold of 0.85 keeps Precision high and false positives low, so only URLs the model is very sure about are blocked. URLs between 0.20 and 0.85 are uncertain: sending them to an analyst keeps Recall high, because stealthy phishing sites are not missed, without blocking legitimate traffic by mistake. Below 0.20, URLs are allowed. This two-threshold policy balances the cost of a missed attack against the cost of a false alarm, and keeps the analysts' workload manageable.

---
## 6 - Conclusion & SOC Engineering Synthesis (Deliverable Solution)

> ### 🛡️ SOC Engineering Synthesis Report
> 1. **Hyperparameter & Sparsity Trade-Off:**

> 2. **Global Interpretability (Odds Ratios & SHAP):**

> 3. **SOC Alert Triage & Dual-Threshold Policy:**
  * *False Positive Triage Note (3 sentences):*

  * *Recommended Threshold Policy:*

In [ ]:

1. **Hyperparameter & Sparsity Trade-Off:** All five models reach Precision and Recall above 0.995 with PR-AUC ≈ 1.0. M3 (L2, C=1.0) has the best raw performance (F1 = 0.9992, 10 FP, 28 FN), but M4 (L1, C=0.01) offers the best trade-off: it removes 24 of 49 features for a small F1 loss (0.9982), at the cost of 18 FP and 72 FN. M2 (strong L2) underfits, with 105 missed phishing URLs, and M5 (balanced weights) barely changes M3 because the classes are only moderately imbalanced.

2. **Global Interpretability (Odds Ratios & SHAP):** The strongest phishing (+) indicators are the URL character ratios: `LetterRatioInURL` (OR ≈ 112), `DegitRatioInURL` (≈ 37) and `SpacialCharRatioInURL` (≈ 37). The strongest benign indicators are `IsHTTPS` (OR ≈ 0.001), `NoOfJS`, `NoOfSelfRef` and `HasSocialNet`. SHAP agrees: `IsHTTPS` has the highest mean |SHAP|, followed by the three character ratios, `NoOfJS` and `HasSocialNet`.

3. **SOC Alert Triage & Dual-Threshold Policy:**
   * *False Positive Triage Note (3 sentences):* The closest false alarm in the 1,000-URL sample (index 330) is a legitimate site that received P(Phishing) = 31.58%, below the 0.50 threshold, so no real false positive occurred in this sample. Its Waterfall plot shows that missing page elements pushed the score toward phishing: `HasSocialNet` = 0 (+1.95), `HasCopyrightInfo` = 0 (+1.21) and `DomainTitleMatchScore` = 0 (+1.10). `IsHTTPS` = 1 (-3.01) and a normal letter ratio pulled it back, which shows that the model reads the absence of typical legitimate elements as suspicious.
   * *Recommended Threshold Policy:* Use τ = 0.85 for automated DNS/firewall blocking, to keep false positives low and avoid blocking legitimate traffic. Send URLs with 0.20 ≤ P(Phishing) < 0.85 to the human analyst queue (threat hunting mode), which keeps Recall high without automatic blocking, and allow URLs below 0.20. Keep τ = 0.50 as the default reference.